RAG
In this section we will start to see the glimpses of RAG. We start by figuring out how to handle external documents. We have already been exposed to building a database in the previous section, and we will use this knowledge to build a database over an example document.

As with data storage, we have many, many options for processing external documents. In this section we will make use of LlamaIndex. The approach will likely vary depending on the nature of your documents - html, pdf, word, folders, etc.

Here, we focus on parsing a single PDF using llama_index and PyMuPDFReader.

In [ ]:
from llama_index.readers.file import PyMuPDFReader
from llama_index.core.node_parser import SentenceSplitter

from pydantic import BaseModel, Field

import fitz

from PIL import Image
import matplotlib.pyplot as plt

import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

import dotenv
import os

from openai import OpenAI

from jinja2 import Environment, FileSystemLoader, select_autoescape
from typing import Any

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
loader = PyMuPDFReader()
documents = loader.load(file_path="data/paper.pdf")

In [ ]:
len(documents)
print(documents[0].text[:1000] + "...")

Extracting images
It is probably handy to have the images extracted from the pdf. This is not always easy to do, but for this paper, we can use PyMuPDF to extract the images. Objects in a pdf are identified by a xref (cross reference) number.

If you know this number, you can extract the image. But how do you find the xref number? One method is use PyMuPDF's image extraction functions. We can just loop through all xrefs and try and extract the image. If it doesn't work, then it's not an image! PyMuPDF will do most of this for us.

In [ ]:
def get_images(path: str):
    doc = fitz.open(path)
    for p, page in enumerate(doc):
        images = page.get_images()
        if len(images) > 0:
            print(f"Page {p} has {len(images)} images")
            for i, img in enumerate(images):
                xref = img[0]
                mref = img[1]
                basepix = fitz.Pixmap(doc,xref)
                maskpix = fitz.Pixmap(doc,mref)
                pix = fitz.Pixmap(basepix, maskpix)
                pix.save(f"./data/page_{p}_image_{i}.png")
    print("Done")

get_images("data/paper.pdf")


In [ ]:
img = Image.open("data/page_4_image_0.png")
plt.imshow(img)
plt.axis("off")
plt.show()

In some cases, this might not be possible. Another method is to convert the pdf pages to images. We can then pass the images to a vision LLM and ask it to extract the images.

Creating a vector database
Now we have our documents, we can create a vector database. We will use Chroma as before.

First, we use the text_parser we created before to split the documents into chunks, and create indices. Essentially, the process is:

Split the document into chunks;
Add the chunks to a list;
Add the chunks to a database, assigning a unique index, and any metadata to the chunks.
The splitting can occur in a few different ways: at ., page-breaks, paragraphs, sentences. You can also choose different chunk sizes and overlap sizes.

We use the SentenceSplitter from LlamaIndex, and just pick some generic parameters.

In [ ]:
def chunker(chunk_size: int, overlap: int, documents: Any) -> tuple[list[str], list[int]]:
    text_parser = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
    )

    text_chunks = []
    doc_idxs = []
    for doc_idx, doc in enumerate(documents):
        cur_text_chunks = text_parser.split_text(doc.text)
        text_chunks.extend(cur_text_chunks)
        doc_idxs.extend([doc_idx] * len(cur_text_chunks))

    return text_chunks, doc_idxs

text_chunks, doc_idxs = chunker(chunk_size=1024, overlap=128, documents=documents)

len(text_chunks)

In [ ]:
class DocumentDB:
    def __init__(self, name: str, model_name: str = "text-embedding-3-small"):
        self.model_name = model_name
        self.client = chromadb.PersistentClient(path="./")
        self.embedding_function = OpenAIEmbeddingFunction(api_key=OPENAI_API_KEY, model_name=model_name)
        self.chat_db = self.client.create_collection(name=name, embedding_function=self.embedding_function, metadata={"hnsw:space": "cosine"})
        self.id_counter = 0


    def add_chunks_to_db(self, chunks: list[str], doc_idxs: list[int], metadata: dict = {}):
        """Add text chunks to the database.

        Args:
            chunks (list[str]): List of text chunks.
            doc_idxs (list[int]): List of corresponding document indices.
        """
        self.chat_db.add(
            documents=chunks,
            metadatas=[{"doc_idx": idx} for idx in doc_idxs],
            ids=[f"chunk_{self.id_counter + i}" for i in range(len(chunks))]
        )
        self.id_counter += len(chunks)


    def get_all_entries(self) -> dict:
        """Grab all of the entries in the database.

        Returns:
            dict: All entries in the database.
        """
        return self.chat_db.get()
    

    def clear_db(self, reinitialize: bool = True):
        """Clear the database of all entries, and reinitialize it.

        Args:
            reinitialize (bool, optional): _description_. Defaults to True.
        """
        self.client.delete_collection(self.chat_db.name)
        # re-initialize the database
        if reinitialize:
            self.__init__(self.chat_db.name, self.model_name)


    def query_db(self, query_text: str, n_results: int = 2) -> dict:
        """Given some query text, return the n_results most similar entries in the database.

        Args:
            query_text (str): The text to query the database with.
            n_results (int): The number of results to return.

        Returns:
            dict: The most similar entries in the database.
        """
        return self.chat_db.query(query_texts=[query_text], n_results=n_results)

In [ ]:
doc_db = DocumentDB("paper_db")
doc_db.add_chunks_to_db(chunks=text_chunks, doc_idxs=doc_idxs)

sample_query = "Abstract"
results = doc_db.query_db(sample_query, n_results=3)
print(f"Sample query results for '{sample_query}':")
results

In [ ]:
client = OpenAI()
def load_template(template_filepath: str, arguments: dict[str, Any]) -> str:
    env = Environment(
        loader=FileSystemLoader(searchpath='./'),
        autoescape=select_autoescape()
    )
    template = env.get_template(template_filepath)
    return template.render(**arguments)

system_prompt = load_template(
    template_filepath="prompts/documents/rag_system_prompt.jinja",
    arguments={}
)

In [ ]:
def combine_context(documents: list[str], scores: list[float]) -> str:
    string = ""
    for document, score in zip(documents, scores):
        string += f"{document}\nCosine distance: {score:.2f}\n{'-'*10}\n"
    return string

def get_context(user_input: str, n_results: int = 2, doc_db: DocumentDB = doc_db) -> str:
    results = doc_db.query_db(user_input, n_results=n_results)
    context_list = results["documents"][0]
    combined_context = combine_context(context_list, results["distances"][0])
    if not combined_context:
        combined_context = "No relevant chat history found."
    return combined_context

query = "What are the main findings of this paper?"
context = get_context(query, n_results=3)
user_prompt = (
    f"Query: {query}\n\n"
    f"Context: {context}"
)

print(user_prompt)

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    stream=True,
    temperature=0.0
)

for chunk in response:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

Evaluate the RAG process

how we can evaluate the RAG process. Well, it's hard. There are a few possible techniques we can use. And here we will demonstrate a few here:

    Perplexity

    Semantic similarity

    Faithfulness

The core of these final two methods (and many methods that evaluate RAG systems) involves feeding the entire paper into an LLM and asking it to generate some questions and some answers based on the paper. We can then assess things like semantic similarity. We can also ask the model to evaluate whether the answer it gave can actually be inferred from the context given.



In [ ]:
from llama_index.readers.file import PyMuPDFReader
from llama_index.core.node_parser import SentenceSplitter

from pydantic import BaseModel, Field

import fitz

from PIL import Image
import matplotlib.pyplot as plt

import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

import dotenv
import os

from openai import OpenAI

from jinja2 import Environment, FileSystemLoader, select_autoescape
from typing import Any
import json

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

Log probabilities
A language model returns a probability distribution over tokens in order to give us an idea of which token to predict next.

It is usually more convenient to work with the logarithm of these probabilities (logprobs) for a few theoretical and practical reasons:

It turns multiplications into additions, which is handy if you want to look at sequences of outputs.

It helps with numerical issues. Multiplying very small numbers could cause underflow in floating point operations. Taking the log converts small numbers to big numbers.

But we can also get the logprobs from OpenAI models (and many other models) in order to develop metrics:

Classification tasks: a measure of confidence in the result;

During RAG: confidence of whether the answer is contained in the retrieved context;

Autocomplete;

Perplexity: overall confidence in a result.

In [ ]:
client = OpenAI()

def get_completion(
    messages: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=512,
    temperature=0,
    stop=None,
    seed=420,
    tools=None,
    logprobs=None,
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stop": stop,
        "seed": seed,
        "logprobs": logprobs,
        "top_logprobs": top_logprobs,
    }

    completion = client.chat.completions.create(**params)

    return completion

In [ ]:
prompt = (
    "You will be given a list of sentences to classify into a particular field of study. "
    "You will need to classify each sentence into one of the following categories:\n"
    "- Physics\n"
    "- Biology\n"
    "- Computer Science\n"
    "Respond only with one of these categories.\n\n"
    "Sentence: {sentence}"
)

In [ ]:
sentences = [
    "Connections between neurons can be mapped by acquiring and analysing electron microscopic brain images.",
    "A straightforward way to quantify the creation of light is through the coefficient of spontaneous emission.",
    "This method optimizes the simulation of protein folding using deep learning.",
]

for sentence in sentences:
    messages = [{"role": "system", "content": prompt.format(sentence=sentence)}]
    completion = get_completion(messages, model="gpt-4o-mini")

    print(f"Sentence: {sentence}\nClassification: {completion.choices[0].message.content}\n")

We can return the top token and the logprobs

In [ ]:
import math

for sentence in sentences:
    messages = [{"role": "system", "content": prompt.format(sentence=sentence)}]
    completion = get_completion(
        messages,
        model="gpt-4o-mini",
        temperature=0.0,
        max_tokens=64,
        logprobs=True,
        top_logprobs=2,
    )
    logprobs = completion.choices[0].logprobs.content[0]

    print(f"Sentence: {sentence}\n"
          f"Classification: {completion.choices[0].message.content}\n"
          f"Logprobs: {math.exp(logprobs.logprob)*100:.2f}\n"
        )

In [ ]:
Perplexity
Perplexity can be considered a measure of uncertainty. In the context of LLMs, it is calculated by taking the average of the logprobs and exponentiating the negative. If we have a tokenized sequence $X = (x_0, x_1, ... x_t), then the perplexity is

 

The 
 term is the log-likelihood of the 
 token conditioned on the preceding tokens before 
. Let's look at an example. To see this in action, we ask two questions: one that has a fairly certain answer, and another that is more speculative.

In [ ]:
questions = [
    "In a few sentences, consicely summarize the theory of special relativity.",
    "In a few sentences, consicely explain who you think will win the 2025 Formula One Drivers' Championship.",
]

import numpy as np

for question in questions:
    messages = [{"role": "system", "content": question}]
    completion = get_completion(messages, model="gpt-4o-mini", logprobs=True, temperature=0.0)

    log_probs = [token.logprob for token in completion.choices[0].logprobs.content]
    response = completion.choices[0].message.content
    perplexity_score = np.exp(-np.mean(log_probs))

    print(
        f"Question: {question}\nAnswer: {completion.choices[0].message.content}\n"
        f"Perplexity: {perplexity_score:.2f}\n")

Evaluating RAG using synthetic data
In these next examples, we look at some methods to evaluate RAG - semantic similarity and faithfulness.

We will use the same approach as previous notebook. So we have moved a bunch of our code into a utils.py file. We have mostly kept things the same, but have a look over it and make sure you understand how it all works.

In [ ]:
from utils import chunker, DocumentDB, load_template

loader = PyMuPDFReader()
documents = loader.load(file_path="data/paper.pdf")
text_chunks, doc_idxs = chunker(chunk_size=1024, overlap=128, documents=documents)

doc_db = DocumentDB("paper_db", path="../data-storage-and-ingestion/")

Generate question answer pairs
For this, we will use gpt-4o because we want high quality question answer pairs. Ideally, you would do this with humans - subject matter experts would carefully hand-craft these pairs.

The first stage is to then generate 10 Q&A pairs using pydantic again. The implementations presented here closely follow the method used by the RAGAS library.

We implement a Pydantic BaseModel class that will house our list of questions.

In [ ]:
class QAPairs(BaseModel):
    questions: list[str] = Field(..., title="List of questions")
    answers: list[str] = Field(..., title="List of answers")

print(QAPairs.model_json_schema())

Next, we need a prompt that we can use to generate these Q&A pairs. It looks something like this:

You are a reading comprehension system that is an expert at extracting information from academic papers.
Your task is to carefully read the provided text "CONTEXT" and then generate question and answer pairs.
Your questions should be concise. Your answers should be as detailed as possible, including any mathematical or numerical results from the text.
You should aim to produce approximately one paragraph for your answers (100-200 words).
Your questions should be a mixture of general, high-level concepts, and also highly detailed questions about specific points, including any mathematical or numerical results.
You should respond in JSON format according to the following schema:

{{ schema }}

You should generate {{ number }} question and answer pairs.


In [ ]:
system_prompt_qa = load_template(
    "prompts/evaluation/qa_generation_system_prompt.jinja",
    {
        "number" : 10,
        "schema" : QAPairs.model_json_schema()
    }
)
print(system_prompt_qa)

Next, we need to the pages of the pdf as a single text string

In [ ]:
pdf_text = " ".join([doc.text for doc in documents])
client = OpenAI()

user_prompt = (
    f"CONTEXT:\n\n{pdf_text}"
)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt_qa},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.1,
    response_format={"type": "json_object"}
)

questions_answers = QAPairs(**json.loads(response.choices[0].message.content))

# save the Q&A to file
with open("data/qa.json", "w") as f:
    json.dump(questions_answers.dict(), f, indent=4)
    
print(questions_answers.questions[0])
print('---')
print(questions_answers.answers[0])

Semantic Similarity
We can now try and do cosine similarity scores between the returned contexts and the actual answers.

In [ ]:
from utils import rag_query

example_query = questions_answers.questions[0]

response, context = rag_query(
    query=example_query,
    n_context=5,
    doc_db=doc_db,
    return_context=True
)

print(response)

In [ ]:
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

client = OpenAI()

response_embedding = client.embeddings.create(
    input=response,
    model="text-embedding-3-small"
).data[0].embedding

answer_embedding = client.embeddings.create(
    input=questions_answers.answers[0],
    model="text-embedding-3-small"
).data[0].embedding


cosine_similarity([response_embedding], [answer_embedding])

Faithfulness
This is a little more complicated. First, we get an LLM to extract key statements from the answer. For example:

[
    ['This study was conducted by Mallinson et al.'],
    ['The main focus is to investigate avalanches and criticality in self-organized nanoscale network.']
    ['They analyzed electrical conductance.']
    ['They analyzed the behavior of the networks under various stimulus conditions.']
]
We then ask a second LLM to look at each statement and see if that statement can be inferred from the text, assigning a score of 0 for no, and 1 for yes.

To do this, we create two additional Pydantic classes:

In [ ]:
class Statements(BaseModel):
    simpler_statements: list[str] = Field(..., description="the simpler statements")


class StatementFaithfulnessAnswer(BaseModel):
    statement: str = Field(..., description="the original statement, word-for-word")
    reason: str = Field(..., description="the reason of the verdict")
    verdict: int = Field(..., description="the verdict(0/1) of the faithfulness.")


class Faithfulness(BaseModel):
    answers: list[StatementFaithfulnessAnswer] = Field(..., description="the faithfulness answers")
    score: float = Field(..., description="the average faithfulness score")

We also create two more prompts statement_instruction, and faithfulness_instruction

Given a piece of text, analyze the complexity of each sentence and break down each sentence into one or more fully understandable statements while also ensuring no pronouns are used in each statement. Format the outputs in JSON, according to the following schema:

{{ schema }}

Here is a new piece of text:

{{ statement }}
Your task is to judge the faithfulness of a statement based on a given context. For the statement you must return verdict as 1 if the statement can be directly inferred based on the context or 0 if the statement can not be directly inferred based on the context.

You will give the exact statement, the reason, and the verdict.

Format the outputs in JSON, according to the following schema:

{{ schema }}

Here is a statement:

{{ statement }}

In [ ]:
def get_statements(answer):
    prompt = load_template(
        "prompts/faithfulness/statement_instruction.jinja",
        {
            "schema" : Statements.model_json_schema(),
            "text" : answer
        }
    )

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": answer}
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
        logprobs=True,
    )

    return Statements(**json.loads(completion.choices[0].message.content))
statements = get_statements(response)
from rich.pretty import pprint
print(response)
pprint(statements)

In [ ]:
def get_faithfulness(statements : Statements, context):
    context_joined = " ".join(context)
    faithfulness_answers = []

    for statement in statements.simpler_statements:
        prompt = load_template(
            "prompts/faithfulness/faithfulness_instruction.jinja",
            {
                "schema" : StatementFaithfulnessAnswer.model_json_schema(),
                "statement" : statement,
                "context" : context_joined
            }
        )

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": context_joined}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        ).choices[0].message.content

        faithfulness_answers.append(StatementFaithfulnessAnswer(**json.loads(response)))

    score = sum([answer.verdict for answer in faithfulness_answers]) / len(faithfulness_answers)

    return Faithfulness(answers=faithfulness_answers, score=score)

results = get_faithfulness(statements, context)
pprint(results)